# 00 · Setup (about 10 minutes)

Run the cells **from top to bottom**: click a cell, then press **Shift + Enter**.
When the smoke test prints **All good!** and the backup check prints **Backup key works.**, you are ready for Session 1.

You need: a Google account, and your two keys from the pre-work (Groq and Google), saved in Colab Secrets.

## 1. Get the workshop code

In [ ]:
# Run this first. In Colab it downloads the workshop code; on a laptop it finds the project folder.
import os, subprocess, sys

REPO_URL = "https://github.com/1Baleid/tuwaiq-agent-workshop.git"   # the student repo
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/workshop"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/workshop"], check=True)
    os.chdir("/content/workshop")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-colab.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # the code lives one folder up
sys.path.insert(0, os.getcwd())
print("Project folder:", os.getcwd())

## 2. Check your keys

In the pre-work you saved `GROQ_API_KEY` and `GEMINI_API_KEY` in Colab Secrets (the **key icon** 🔑 in the
left bar), with **Notebook access** on. If you have not done that yet, do it now (pre-work, Step 4).
Never paste a key into a code cell: anyone who sees the notebook could use it.

Then run the next cell.

In [ ]:
import config, llm

provider = config.PROVIDERS[config.LLM_PROVIDER]
llm.get_api_key(provider["key_name"])   # stops with a clear message if the key is missing
print(f"Key found. Provider: {config.LLM_PROVIDER}, model: {provider['model']}")

## 3. Smoke test

Checks your Python packages, downloads the embedding model (~90 MB, once) and makes one LLM call.

In [ ]:
import scripts.smoke_test as smoke

exit_code = smoke.main(["--quick"])   # 0 means every check passed

## 4. Check your backup key (Google)

If the main provider has problems on the day, we switch to Google. This checks that key now.

In [ ]:
import config, llm

main_provider = config.LLM_PROVIDER
config.LLM_PROVIDER = "gemini"                 # try the backup provider for one call
try:
    reply = llm.chat([{"role": "user", "content": "Reply with just the word OK."}])
    print("Backup key works. Gemini replied:", reply.strip()[:30])
finally:
    config.LLM_PROVIDER = main_provider        # switch back

---
## Plan B: a local model (only when the instructor asks)

If the online LLM is down or too slow, we switch to a small open model that runs inside **your** Colab.

1. **Runtime → Change runtime type → T4 GPU → Save.** This restarts Colab, so run cell 1 again first.
2. Run the cell below. It takes about 3–5 minutes (it downloads ~4 GB inside Google's network, not over the room Wi-Fi).

In [ ]:
import config
import scripts.local_model as local_model
import scripts.smoke_test as smoke

local_model.start()
config.LLM_PROVIDER = "local"     # the ONE switch: everything now uses the local model
exit_code = smoke.main(["--quick"])

---
## Stuck?

* Look up the error message in the pre-work guide ("If something goes wrong"), or ask a helper.
  At home: send a screenshot to the organizers.
* Stuck on an exercise for more than 10 minutes? Get the working version of that file, e.g.:
  `!git checkout origin/solutions -- rag.py`